<span style='color:#0066cc'> <span style='font-family:serif'> <font size="15"> **Access Gridded Ozone Data from TEMPO**

Total ozone Level 3 files provide ozone information on a regular grid covering the TEMPO field of regard for nominal TEMPO observations. Level 3 files are derived by combining information from all Level 2 files constituting a TEMPO East-West scan cycle. The files are provided in netCDF4 format, and contain information on total column ozone and some auxiliary derived and ancillary input parameters including effective cloud fraction, effective cloud pressure, radiative cloud fraction, SO2 index, and terrain pressure. The re-gridding algorithm uses an area-weighted approach. These data reached provisional validation on December 9, 2024. **Source**: [NASA Earthdata](https://www.earthdata.nasa.gov/data/catalog/larc-cloud-tempo-o3tot-l3-v03)


<span style='color:#ff6666'><font size="5"> **Requirements**
1. <font size="3"> Valid Earthdata Login (EDL)

<span style='color:#ff6666'><font size="5"> **Objectives**

### Subset a remote file

- <font size="3">**a)** By Variables
- <font size="3">**b)** By Spatial selection

### Subset multiple remote files

- <font size="3">Stream subset of data

<span style='color:#ff6666'><font size="5"> **References**


<font size="3"> **Suleiman, R. (2024)**. <i>TEMPO gridded ozone total column V03 (BETA)</i> [Data set]. NASA Langley Atmospheric Science Data Center Distributed Active Archive Center. https://doi.org/10.5067/IS-40E/TEMPO/O3TOT_L3.003 Date Accessed: 2026-04-07

In [ ]:
import xarray as xr
import datetime as dt
import earthaccess

# import pydap-specific tools
from pydap.client import get_cmr_urls, open_url
from pydap.client import to_netcdf as dap_to_netcdf

# Finding OPeNDAP URLs

<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **Query opendap urls using NASA's CMR API**

In [ ]:
TEMPO_O3TOT_L3_V03_ccid = "C2930764281-LARC_CLOUD"
time_range = [dt.datetime(2025, 9, 1), dt.datetime(2025, 9, 30)] # One month of data

cmr_urls = get_cmr_urls(ccid=TEMPO_O3TOT_L3_V03_ccid, time_range=time_range, limit=1000) # you can incread the limit of results

print("################################################ \n We found a total of ", len(cmr_urls), "OPeNDAP URLS!!!\n################################################")

In [ ]:
cmr_urls[0]

<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **EDL Authentication via earthaccess and OPeNDAP**

<font size="3.5"> You can authenticate via earthaccess as demonstrated below. You must have a valid EDL account. There are two strategies for authenticating with `earthaccess`:

1. `strategy="interactive"`. This will promt your edl username-password.
2. `strategy="netrc"`. Use this if the notebook is running on an environment where a `.netrc` with your credentials is recoverable. T

<font size="3.5"> Below the default will be `netrc`, assuming the user has executed the notebook **Authenticate.ipynb**. If not, you can change the strategy to `"interactive"`.



In [ ]:
from earthaccess.exceptions import LoginStrategyUnavailable
try:
    auth = earthaccess.login(strategy="netrc", persist=True) # you will be promted to add your EDL credentials
except LoginStrategyUnavailable:
    auth = earthaccess.login(strategy="interactive", persist=True)

# pass Token Authorization to a new Session.
my_session = session=auth.get_session()


<span style='font-family:serif'> <font size="5.5"><span style='color:#0066cc'> **Accessing Metadata-ONLY with PyDAP**

<font size="3"> We can access <span style='color:#ff6666'>**OPeNDAP**</span>-produced metadata to identify the variables of interest. In particular those associated with latitude and longitude values

<font size="3"> Below need to request the <span style='color:#ff6666'>**DAP4**</span> metadata from the remote server. To specify the protocol, we have 2 options:

**1.** <font size="3"> Replace "https" with "dap4" in the URL. This works when using `Xarray`:
```python
open_url(url.replace("https","dap4"), ...)
```
**2**. <font size="3"> Specify the protocol directly (**does not work with Xarray**)
```python
open_url(url, protocol='dap4', ...)
```

<font size="3"> Below we follow option **1)** above.


In [ ]:
%%time
pyds = open_url(cmr_urls[0].replace("https","dap4"), session=my_session)
pyds.tree()

<font size="3.5"> Having now the full list of variables, we are interested in:
* <font size="3.5"> Geographic location data,
* <font size="3.5"> The variable `weight`.
* <font size="3.5"> Time.
* <font size="3.5"> All of the data inside the group `support_data`
* <font size="3.5"> The variable `o3_below_cloud` for further processing.


<font size="3.5"> The remote file contains hierarchies such as Groups, which act like folder in a filesystem. In DAP4 we can define variables across different hierarchies, by defining their `fully qualifying name`, analogous to a file inside a filesystem. 

<font size="3.5"> The list of variables we are interested in are declared in `keep_variables`, below


In [ ]:
keep_variables = ['/longitude', '/latitude', '/time', "/support_data", "/product/o3_below_cloud"]

### Subset by coordinate values

<font size="3.5"> Below we use Xarray to download coordinate data using PyDAP in the background. THe goal is to identify a region of interest and only download data inside that region of interest.



In [ ]:
# define bounding box by edges
lat_min, lat_max = 45, 60
lon_min, lon_max = -123, -120.5

ds = xr.open_dataset(cmr_urls[0].replace("https","dap4"), session=my_session, engine='pydap')

lat_index = ds.latitude.to_index()
lon_index = ds.longitude.to_index()

lat_start = lat_index.get_indexer([lat_min], method="nearest")[0]
lat_end   = lat_index.get_indexer([lat_max], method="nearest")[0]

lon_start = lon_index.get_indexer([lon_min], method="nearest")[0]
lon_end   = lon_index.get_indexer([lon_max], method="nearest")[0]

# define slicing argument to use in PyDAP to stream the subset of data
dim_slices = {'latitude': (lat_start, lat_end), 'longitude': (lon_start, lon_end)}


### Stream data 

In [ ]:
%%time
dap_to_netcdf(cmr_urls[:30], session=my_session, output_path = "./data", dim_slices=dim_slices, keep_variables=keep_variables)

## check the data

In [ ]:
%%time
ds1 = xr.open_mfdataset("data/TEMPO_O3*.nc4", group="/", parallel=True)
ds2 = xr.open_mfdataset("data/TEMPO_O3*.nc4", group="/support_data", parallel=True, concat_dim='time', combine='nested')
ds3 = xr.open_mfdataset("data/TEMPO_O3*.nc4", group="/product", parallel=True, concat_dim='time', combine='nested')
ds = xr.merge([ds1, ds2, ds3])
ds

In [ ]:
ds['terrain_height'].isel(time=0).plot();